# Gold Layer — Taxi Analytics

Business-ready analytics table for the taxi booking dataset. Reads from `silver_taxi_data` in the `team-1-taxi-service` pipeline.  
Builds `gold_taxi_data` — one row per booking with derived measures, status/validity flags, decoded capabilities, and calendar attributes.

> **Note:** The star schema (`fact_booking` + 8 dimensions) has been removed from the pipeline. Code is preserved in Git history.

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql import Row

In [0]:
# ---------------------------------------------------------------------------
# IQR-based price outlier threshold: Q3 + 3*IQR = 8.16 per mile
# Derived from analysis of completed trips with distance > 0
# ---------------------------------------------------------------------------
_PRICE_PER_MILE_UPPER_FENCE = 8.16


# ---------------------------------------------------------------------------
# gold_taxi_data — Business-ready analytics table
# ---------------------------------------------------------------------------
@dlt.table(
    name="gold_taxi_data",
    comment="Business-ready analytics table — one row per booking with derived measures, status/validity flags, decoded capability flags, and calendar attributes"
)
def gold_taxi_data():
    """
    Reads from silver_taxi_data (one row per booking_id, verified unique).
    All derived columns live here rather than Silver, per §5.2/§5.3 of the Project Brief.

    Derived column groups:
      1. Duration measures (minutes): wait_time, trip_duration, total_time_taken
      2. Price: price_per_mile
      3. Status flags: is_completed, is_cancelled, is_no_fare
      4. Validity flags: is_valid_distance, is_valid_timing, is_valid_duration, is_price_outlier
      5. Decoded capability flags (13 booleans from the Silver-decoded capabilities string)
      6. Calendar attributes derived from pickup_due: year, month, month_name, day_name, is_weekend
    """
    silver = dlt.read("silver_taxi_data")

    return silver.select(
        # --- Pass-through columns from Silver ---
        silver["booking_id"],
        silver["trip_status"],
        silver["pickup_due"],
        silver["completed"],
        silver["time_dispatched"],
        silver["time_vehicle_arrived"],
        silver["time_picked_up"],
        silver["driver"],
        silver["vehicle"],
        silver["price"],
        silver["distance"],
        silver["priority"],
        silver["payment_type"],
        silver["booking_source"],
        silver["booked_by"],
        silver["capabilities"],
        silver["pickup_zone"],
        silver["destination_zone"],
        silver["pickup_latitude"],
        silver["pickup_longitude"],
        silver["destination_latitude"],
        silver["destination_longitude"],
        silver["source_file"],
        silver["ingestion_timestamp"],

        # --- Duration measures (minutes) ---
        F.round(
            (F.unix_timestamp("time_vehicle_arrived") - F.unix_timestamp("time_dispatched")) / 60, 2
        ).alias("wait_time_minutes"),
        F.round(
            (F.unix_timestamp("completed") - F.unix_timestamp("time_picked_up")) / 60, 2
        ).alias("trip_duration_minutes"),
        F.round(
            (F.unix_timestamp("completed") - F.unix_timestamp("time_dispatched")) / 60, 2
        ).alias("total_time_taken"),

        # --- Price derived ---
        F.when(
            F.col("distance") > 0,
            F.round(F.col("price") / F.col("distance"), 2)
        ).otherwise(F.lit(None)).alias("price_per_mile"),

        # --- Status flags ---
        (F.col("trip_status") == "Completed").cast("boolean").alias("is_completed"),
        (F.col("trip_status") == "Cancelled").cast("boolean").alias("is_cancelled"),
        (F.col("trip_status") == "No Fare").cast("boolean").alias("is_no_fare"),

        # --- Validity flags ---
        (F.col("distance") > 0).cast("boolean").alias("is_valid_distance"),
        (
            F.col("time_vehicle_arrived").isNotNull() & F.col("time_picked_up").isNotNull()
        ).cast("boolean").alias("is_valid_timing"),
        (
            (F.col("trip_status") != "Completed") |
            ((F.unix_timestamp("completed") - F.unix_timestamp("pickup_due")) >= 0)
        ).cast("boolean").alias("is_valid_duration"),
        F.when(
            (F.col("distance") > 0) & ((F.col("price") / F.col("distance")) > _PRICE_PER_MILE_UPPER_FENCE),
            F.lit(True)
        ).otherwise(F.lit(False)).alias("is_price_outlier"),

        # --- Decoded capability flags ---
        # Each flag checks the Silver-decoded comma-separated capabilities string
        # Codes: Z=Card Reader, D=Delivery, H=High Car, L=Low Car, W=Wheelchair,
        #        M=Minibus, F=Female Driver, V=VIP, T=Tour, P=Pet, 6/7/8=Seater
        F.coalesce(F.col("capabilities").contains("Card Reader"), F.lit(False)).alias("has_card_reader"),
        F.coalesce(F.col("capabilities").contains("Delivery"), F.lit(False)).alias("has_delivery"),
        F.coalesce(F.col("capabilities").contains("High Car"), F.lit(False)).alias("has_high_car"),
        F.coalesce(F.col("capabilities").contains("Low Car"), F.lit(False)).alias("has_low_car"),
        F.coalesce(F.col("capabilities").contains("Wheelchair"), F.lit(False)).alias("has_wheelchair"),
        F.coalesce(F.col("capabilities").contains("Minibus"), F.lit(False)).alias("has_minibus"),
        F.coalesce(F.col("capabilities").contains("Female"), F.lit(False)).alias("has_female_driver"),
        F.coalesce(F.col("capabilities").contains("VIP"), F.lit(False)).alias("has_vip"),
        F.coalesce(F.col("capabilities").contains("Tour"), F.lit(False)).alias("has_tour"),
        F.coalesce(F.col("capabilities").contains("Pet"), F.lit(False)).alias("has_pet"),
        F.coalesce(F.col("capabilities").contains("6 seater"), F.lit(False)).alias("has_six_seater"),
        F.coalesce(F.col("capabilities").contains("7 seater"), F.lit(False)).alias("has_seven_seater"),
        F.coalesce(F.col("capabilities").contains("8 seater"), F.lit(False)).alias("has_eight_seater"),

        # --- Calendar attributes (derived from pickup_due) ---
        F.year("pickup_due").alias("year"),
        F.month("pickup_due").alias("month"),
        F.date_format("pickup_due", "MMMM").alias("month_name"),
        F.date_format("pickup_due", "EEEE").alias("day_name"),
        (F.dayofweek("pickup_due").isin(1, 7)).cast("boolean").alias("is_weekend"),
    )

In [0]:
%sql
-- Step 4a: Row count matches Silver, booking_id is unique
SELECT
  COUNT(*)                                    AS total_rows,
  COUNT(DISTINCT booking_id)                  AS distinct_booking_ids,
  COUNT(*) - COUNT(DISTINCT booking_id)       AS duplicates
FROM `students_data`.`team-1-data-schema`.silver_taxi_data

In [0]:
%sql
-- Step 4b: Spot-check duration measures and status flags against source timestamps
SELECT
  booking_id,
  trip_status,

  -- Duration inputs
  time_dispatched,
  time_vehicle_arrived,
  time_picked_up,
  completed,

  -- Expected Gold derivations
  ROUND((unix_timestamp(time_vehicle_arrived) - unix_timestamp(time_dispatched)) / 60, 2)  AS calc_wait_time_minutes,
  ROUND((unix_timestamp(completed) - unix_timestamp(time_picked_up)) / 60, 2)              AS calc_trip_duration_minutes,
  ROUND((unix_timestamp(completed) - unix_timestamp(time_dispatched)) / 60, 2)             AS calc_total_time_taken,

  -- Status flags
  (trip_status = 'Completed')  AS calc_is_completed,
  (trip_status = 'Cancelled')  AS calc_is_cancelled,
  (trip_status = 'No Fare')    AS calc_is_no_fare,

  -- Price
  price,
  distance,
  CASE WHEN distance > 0 THEN ROUND(price / distance, 2) END AS calc_price_per_mile

FROM `students_data`.`team-1-data-schema`.silver_taxi_data
WHERE time_dispatched IS NOT NULL
  AND time_vehicle_arrived IS NOT NULL
  AND completed IS NOT NULL
LIMIT 10

In [0]:
%sql
-- Step 4c: Verify capability flag decode against the decoded capabilities string
SELECT
  capabilities,
  capabilities LIKE '%Card Reader%'  AS has_card_reader,
  capabilities LIKE '%Delivery%'     AS has_delivery,
  capabilities LIKE '%High Car%'     AS has_high_car,
  capabilities LIKE '%Low Car%'      AS has_low_car,
  capabilities LIKE '%Minibus%'      AS has_minibus,
  capabilities LIKE '%VIP%'          AS has_vip,
  capabilities LIKE '%Female%'       AS has_female_driver,
  capabilities LIKE '%Tour%'         AS has_tour,
  capabilities LIKE '%Pet%'          AS has_pet,
  capabilities LIKE '%6 seater%'     AS has_six_seater,
  capabilities LIKE '%7 seater%'     AS has_seven_seater,
  capabilities LIKE '%8 seater%'     AS has_eight_seater
FROM (
  SELECT DISTINCT capabilities
  FROM `students_data`.`team-1-data-schema`.silver_taxi_data
  WHERE capabilities IS NOT NULL
)
ORDER BY capabilities
LIMIT 15